# 03: Feature Engineering and Feature Selection (USA Real Estate)

In this notebook, we engineer new features for the USA real estate dataset and
evaluate whether these features provide **statistically meaningful signal** for
predicting the target variable `price`.

We will:
- Design and create engineered features (ratios, intensities, location effects).
- Test whether each engineered feature is associated with `price` beyond chance.
- Compare engineered features against their raw “parent” features.
- Perform feature selection and **drop raw parents** where engineered variants
  perform better, to avoid redundancy and leakage.


---
# 1. Imports and Global Configuration

In this section, we import the required libraries and define global constants
for the feature engineering workflow.

We will rely on:

- `pandas` / `numpy` for data manipulation
- `scipy.stats` for statistical tests
- `sklearn.feature_selection` for correlation-based feature selection
- `sklearn.preprocessing` for scaling
- `sklearn.linear_model` for regularised models (optional signal check)

**Pearson Correlation Coefficient**

For a numeric feature \( X \) and target \( Y \), the Pearson correlation
coefficient is:

$$
r = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}
         {\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2}
          \sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}}
$$

A two-sided p-value for \( r \) allows us to test:

- \( H_0 \): no linear association between \( X \) and \( Y \)
- \( H_1 \): there is a linear association between \( X \) and \( Y \)


In [ ]:
"""Global configuration and imports for the feature engineering notebook.

This module sets up common configuration for the USA property pricing
regression task and mounts Google Drive when running in Google Colab.

Attributes:
    TARGET_COL (str): Name of the supervised learning target column.
    RANDOM_STATE (int): Seed used for reproducible operations.
    MAX_SAMPLE_FOR_STATS (int): Maximum number of rows to sample for
        expensive statistical analyses.
"""

import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from google.colab import drive  # type: ignore
from scipy import stats
from sklearn.feature_selection import f_regression, mutual_info_regression
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
from datetime import datetime

# ---- Drive mount (Colab) ----
drive.mount("/content/drive")

warnings.filterwarnings("ignore")

TARGET_COL: str = "price"
RANDOM_STATE: int = 42
MAX_SAMPLE_FOR_STATS: int = 200_000

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The libraries and configuration above give us all the core tools needed for
feature engineering, statistical testing, and feature selection. We also cap
the sample size for heavy statistical routines to maintain reasonable runtime
for the 2M+ row dataset.

---
# 2. Load Cleaned Dataset

In this section, we load the **cleaned dataset** produced by the data cleaning
pipeline. This dataset should already have:

- Duplicates removed
- Missing values imputed
- Dtypes normalised
- Outlier flags (if any)

We will treat this as the base input for feature engineering.

In [ ]:
"""Dataset loading and post-cleaning column reduction.

This section loads the already-cleaned USA real estate dataset and removes
columns that are not required for downstream feature engineering or modelling.

The column removal list includes:
    • Raw string attributes (e.g., street, zip_code)
    • Missingness indicator flags
    • Log-transformed variants
    • Outlier flags (univariate, multivariate, contextual)

These fields were useful during the cleaning pipeline but must not be included
in the modelling stage to ensure a clean, leakage-free feature space.
"""

import pandas as pd

# ---- Load Cleaned Dataset ----
df_clean = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/DOAA/data_processed/usa_real_estate_clean.csv"
)

assert TARGET_COL in df_clean.columns, (
    "Target column 'price' not found in the cleaned dataset."
)

# Columns produced during cleaning but not needed for feature engineering/model training
COLS_TO_DROP = [
    # Raw attributes removed earlier in the pipeline
    "brokered_by",
    "street",
    "zip_code",
    "prev_sold_date",
    # Missingness indicator flags
    "brokered_by_was_missing",
    "bed_was_missing",
    "acre_lot_was_missing",
    "street_was_missing",
    "city_was_missing",
    "state_was_missing",
    "zip_code_was_missing",
    "house_size_was_missing",
    "bath_was_missing",
    "prev_sold_date_was_missing",
    # Log transforms
    "price_log",
    "house_size_log",
    "bed_log",
    "bath_log",
    "acre_lot_log",
    # Outlier flags
    "price__is_uni_outlier",
    "house_size__is_uni_outlier",
    "bed__is_uni_outlier",
    "bath__is_uni_outlier",
    "acre_lot__is_uni_outlier",
    "is_multivar_outlier",
    "price_peer_z",
    "is_price_context_outlier",
]

# Drop only columns that exist (safe in case earlier pipeline changed)
existing_drop_cols = [col for col in COLS_TO_DROP if col in df_clean.columns]
df_clean = df_clean.drop(columns=existing_drop_cols)

print(f"[INFO] Cleaned dataset loaded. Final shape: {df_clean.shape}")

[INFO] Cleaned dataset loaded. Final shape: (2224841, 8)


We confirm that the cleaned dataset is available and that the target column
`price` exists. This dataset now forms the foundation for constructing new
engineered features.

---
# 3. Feature Engineering Functions

Here we define reusable functions that construct engineered features from the
cleaned dataset. Below are the formulas for every engineered feature in the dataset.

---

**Total Rooms**

**Formula:**  
$$TR = b + ba$$

Where:  
- $b$  = number of bedrooms  
- $ba$ = number of bathrooms  

---

**Ratio / Intensity Features**

**Bed per Bath**  
$$bed\_per\_bath = b / ba$$

**Bath per Bed**  
$$bath\_per\_bed = ba / b$$

**House Size per Bed**  
$$house\_size\_per\_bed = hs / b$$

**House Size per Bath**  
$$house\_size\_per\_bath = hs / ba$$

**House Size per Room**  
$$house\_size\_per\_room = hs / (b + ba)$$

Where:  
- $hs$ = house_size  
- $b$  = bed  
- $ba$ = bath  

---

**Lot Features**

**Lot Size (in square feet)**  
$$lot\_sqft = a * 43560$$

**Building Coverage Ratio**  
$$building\_coverage\_ratio = hs / lot\_sqft$$
Equivalent to:  
$$building\_coverage\_ratio = hs / (a * 43560)$$

Where:  
- $a$ = acre_lot  
- $hs$ = house_size  

---

**Log1p Transformations**

These use natural logarithm.

**House Size**  
$$house\_size\_log1p = log(1 + hs)$$

**Bed**  
$$bed\_log1p = log(1 + b)$$

**Bath**  
$$bath\_log1p = log(1 + ba)$$

**Acre Lot**  
$$acre\_lot\_log1p = log(1 + a)$$

---

**Density Features (City / State)**

These measure how many listings exist within the same region.

**City Listing Count**  
$$city\_listing\_count(i) = \sum\_{j=1}^{N} 1(city\_j = city\_i)$$

**State Listing Count**  
$$state\_listing\_count(i) = \sum\_{j=1}^{N} 1(state\_j = state\_i)$$

Where:  
- $i$ = current row  
- $j$ = any row  
- $1(condition)$ = indicator function (1 if true, 0 if false)  
- $N$ = total rows  

---

**Status One-Hot Encodings**

**For Sale**  
$status\_is\_for\_sale(i) = 1$ if status = "for_sale", else $0$

**Sold**  
$status\_is\_sold(i) = 1$ if status = "sold", else $0$

**Ready to Build**  
$status\_is\_ready\_to\_build(i) = 1$ if status = "ready\_to\_build", else $0$

---

With the relevant mathematical formulas indicated above, we will proceed to conduct the feature engineering in the code cell below.

In [ ]:
def _safe_div(n: pd.Series, d: pd.Series) -> pd.Series:
    """Perform element-wise safe division between two Series.

    Zeros in the denominator are replaced with NaN before division to avoid
    division-by-zero warnings and infinite values.

    Args:
        n: Numerator series.
        d: Denominator series.

    Returns:
        A pandas Series containing the result of the safe division.
    """
    d_safe = d.replace(0, np.nan)
    return n / d_safe


def engineer_features_minimal(
    df: pd.DataFrame,
    target_col: str = "price",
) -> Tuple[pd.DataFrame, Dict[str, List[str]]]:
    """Engineer leakage-free features for the real-estate dataset.

    This function creates a minimal, interpretable set of engineered features
    from the raw columns. It does not use the target column in any computation
    to strictly avoid target leakage. The function also returns a parent-map
    dictionary that records which raw columns were used to derive each
    engineered feature, supporting downstream transparency and governance.

    Args:
        df: Input cleaned DataFrame containing at least the core real-estate
            features (e.g., ``bed``, ``bath``, ``house_size``, ``acre_lot``,
            ``city``, ``state``, ``status``) where available.
        target_col: Name of the target column. The target is preserved in the
            returned DataFrame but never used to engineer features.

    Returns:
        A tuple of:
            df_fe: DataFrame with engineered features added on top of the
                original columns.
            parent_map: Mapping from engineered feature names to a list of
                original parent columns used to derive them.
    """
    df_fe = df.copy()
    parent_map: Dict[str, List[str]] = {}

    # --- Basic presence flags ---
    has_h = "house_size" in df_fe.columns
    has_bed = "bed" in df_fe.columns
    has_bath = "bath" in df_fe.columns
    has_acre = "acre_lot" in df_fe.columns

    # 1) Total Rooms
    if has_bed and has_bath:
        df_fe["total_rooms"] = df_fe["bed"] + df_fe["bath"]
        parent_map["total_rooms"] = ["bed", "bath"]

    # 2) Ratio Features
    if has_bed and has_bath:
        df_fe["bed_per_bath"] = _safe_div(df_fe["bed"], df_fe["bath"])
        df_fe["bath_per_bed"] = _safe_div(df_fe["bath"], df_fe["bed"])
        parent_map["bed_per_bath"] = ["bed", "bath"]
        parent_map["bath_per_bed"] = ["bed", "bath"]

    if has_h and has_bed:
        df_fe["house_size_per_bed"] = _safe_div(
            df_fe["house_size"],
            df_fe["bed"],
        )
        parent_map["house_size_per_bed"] = ["house_size", "bed"]

    if has_h and has_bath:
        df_fe["house_size_per_bath"] = _safe_div(
            df_fe["house_size"],
            df_fe["bath"],
        )
        parent_map["house_size_per_bath"] = ["house_size", "bath"]

    if has_h and has_bed and has_bath:
        total_rooms = df_fe["bed"] + df_fe["bath"]
        df_fe["house_size_per_room"] = _safe_div(
            df_fe["house_size"],
            total_rooms,
        )
        parent_map["house_size_per_room"] = ["house_size", "bed", "bath"]

    # 3) Lot Features
    if has_acre:
        df_fe["lot_sqft"] = df_fe["acre_lot"] * 43_560
        parent_map["lot_sqft"] = ["acre_lot"]

        if has_h:
            df_fe["building_coverage_ratio"] = _safe_div(
                df_fe["house_size"],
                df_fe["lot_sqft"],
            )
            parent_map["building_coverage_ratio"] = ["house_size", "acre_lot"]

    # 4) Log1p transforms
    for col in ["house_size", "bed", "bath", "acre_lot"]:
        if col in df_fe.columns:
            df_fe[f"{col}_log1p"] = np.log1p(df_fe[col].clip(lower=0))
            parent_map[f"{col}_log1p"] = [col]

    # 5) City/State Density
    if "city" in df_fe.columns:
        df_fe["city_clean"] = (
            df_fe["city"].astype(str).str.lower().str.strip()
        )
        df_fe["city_listing_count"] = (
            df_fe.groupby("city_clean")["city_clean"].transform("count")
        )
        parent_map["city_listing_count"] = ["city"]

    if "state" in df_fe.columns:
        df_fe["state_clean"] = (
            df_fe["state"].astype(str).str.lower().str.strip()
        )
        df_fe["state_listing_count"] = (
            df_fe.groupby("state_clean")["state_clean"].transform("count")
        )
        parent_map["state_listing_count"] = ["state"]

    # 6) Status Encoding — sold / for_sale / ready_to_build
    if "status" in df_fe.columns:
        df_fe["status_clean"] = (
            df_fe["status"].astype(str).str.lower().str.strip()
        )

        df_fe["status_is_for_sale"] = (
            df_fe["status_clean"] == "for_sale"
        ).astype(int)
        df_fe["status_is_sold"] = (
            df_fe["status_clean"] == "sold"
        ).astype(int)
        df_fe["status_is_ready_to_build"] = (
            df_fe["status_clean"] == "ready_to_build"
        ).astype(int)

        parent_map["status_is_for_sale"] = ["status"]
        parent_map["status_is_sold"] = ["status"]
        parent_map["status_is_ready_to_build"] = ["status"]

    return df_fe, parent_map

The `engineer_features` function centralises all feature creation and returns a
`parent_map` dictionary. This mapping will be crucial when we perform feature
selection, because it allows us to **drop raw parent columns** whenever the
engineered feature is preferred.

---
# 4. Apply Feature Engineering

We now call `engineer_features` on the cleaned dataset to generate the extended
feature set and capture the parent relationships between raw and engineered
features.

In [ ]:
def run_minimal_feature_engineering(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict[str, List[str]]]:
    """Apply minimal feature engineering and log shape changes.

    This helper wraps ``engineer_features_minimal`` to:
      * Apply feature engineering on the cleaned dataset.
      * Print pre- and post-engineering shapes.
      * Display the engineered features alongside their parent columns.

    Args:
        df: Cleaned input DataFrame prior to feature engineering.

    Returns:
        A tuple of:
            df_fe: DataFrame with engineered features added.
            parent_map: Mapping from engineered feature names to their
                originating raw columns.
    """
    df_fe, parent_map = engineer_features_minimal(df)

    print(f"[INFO] Shape before feature engineering: {df.shape}")
    print(f"[INFO] Shape after  feature engineering: {df_fe.shape}")
    print("\n[INFO] Engineered features and their parent columns:")
    for feature_name, parents in parent_map.items():
        print(f"  {feature_name}: {parents}")

    return df_fe, parent_map


# ---- Execute Feature Engineering Step ----
df_fe, parent_map = run_minimal_feature_engineering(df_clean)

[INFO] Shape before feature engineering: (2224841, 8)
[INFO] Shape after  feature engineering: (2224841, 28)

[INFO] Engineered features and their parent columns:
  total_rooms: ['bed', 'bath']
  bed_per_bath: ['bed', 'bath']
  bath_per_bed: ['bed', 'bath']
  house_size_per_bed: ['house_size', 'bed']
  house_size_per_bath: ['house_size', 'bath']
  house_size_per_room: ['house_size', 'bed', 'bath']
  lot_sqft: ['acre_lot']
  building_coverage_ratio: ['house_size', 'acre_lot']
  house_size_log1p: ['house_size']
  bed_log1p: ['bed']
  bath_log1p: ['bath']
  acre_lot_log1p: ['acre_lot']
  city_listing_count: ['city']
  state_listing_count: ['state']
  status_is_for_sale: ['status']
  status_is_sold: ['status']
  status_is_ready_to_build: ['status']


We have successfully augmented the dataset with engineered features. The
`parent_map` output summarises which raw columns contributed to each new
feature, setting up the subsequent steps for **impact analysis** and
**redundancy removal**.

---
# 5. Statistical Impact Analysis of Engineered Features

To check whether our engineered features carry meaningful signal for `price`,
we apply two complementary statistical measures on a sampled subset:

1. **Pearson correlation** \( r \) and two-sided p-value.
2. **Mutual Information** (MI), which captures general dependency (not just
   linear).

For a feature \( X \) and target \( Y \), the mutual information is:

$$
I(X; Y) = \sum_{x, y} p(x, y) \log
\left(
\frac{p(x, y)}{p(x)p(y)}
\right)
$$

Higher MI suggests a stronger dependence between the feature and the target.

In [ ]:
def evaluate_engineered_features(
    df: pd.DataFrame,
    target_col: str,
    engineered_cols: List[str],
    max_sample: int = MAX_SAMPLE_FOR_STATS,
) -> pd.DataFrame:
    """Evaluate engineered features using correlation and mutual information.

    This function:
      * Cleans the dataset of infinite and missing values.
      * Optionally subsamples rows to a maximum size for efficiency.
      * Computes Pearson correlation (r and p-value) between each engineered
        feature and the target.
      * Computes mutual information (MI) between each engineered feature and
        the target in a single batch call.

    Args:
        df: Input DataFrame containing the target column and engineered
            feature columns.
        target_col: Name of the numeric target column for regression.
        engineered_cols: List of engineered feature names to evaluate.
        max_sample: Maximum number of rows to use for statistical analysis.
            If the cleaned dataset exceeds this size, a random sample is
            drawn using the global RANDOM_STATE.

    Returns:
        A DataFrame where each row corresponds to one engineered feature and
        includes:
            * ``feature``: Feature name.
            * ``pearson_r``: Pearson correlation coefficient with the target.
            * ``pearson_p``: Two-sided p-value for the Pearson test.
            * ``mutual_info``: Mutual information score with the target.
        The output is sorted in descending order of ``mutual_info``.
    """
    # Keep only target + engineered features and remove invalid values
    df_eval = df[[target_col] + engineered_cols].copy()
    df_eval = df_eval.replace([np.inf, -np.inf], np.nan).dropna()

    if len(df_eval) > max_sample:
        df_eval = df_eval.sample(max_sample, random_state=RANDOM_STATE)

    y = df_eval[target_col].values
    summary_rows: List[Dict[str, float]] = []

    # Pearson correlation for each engineered feature
    for col in engineered_cols:
        x = df_eval[col].values
        r_val, p_val = stats.pearsonr(x, y)
        summary_rows.append(
            {
                "feature": col,
                "pearson_r": r_val,
                "pearson_p": p_val,
            }
        )

    # Mutual information computed in one go for consistency
    X = df_eval[engineered_cols].values
    mi_vals = mutual_info_regression(
        X,
        y,
        random_state=RANDOM_STATE,
    )

    for row, mi in zip(summary_rows, mi_vals):
        row["mutual_info"] = mi

    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.sort_values("mutual_info", ascending=False)

    return summary_df


engineered_cols = list(parent_map.keys())
impact_df = evaluate_engineered_features(
    df_fe,
    target_col=TARGET_COL,
    engineered_cols=engineered_cols,
)

impact_df

,feature,pearson_r,pearson_p,mutual_info
8,house_size_log1p,0.229935,0.000000e+00,0.246755
3,house_size_per_bed,0.270580,0.000000e+00,0.211205
5,house_size_per_room,0.177040,0.000000e+00,0.200019
13,state_listing_count,0.162516,0.000000e+00,0.186109
12,city_listing_count,0.074748,1.111967e-245,0.177282
7,building_coverage_ratio,0.097956,0.000000e+00,0.169887
4,house_size_per_bath,-0.011489,2.772971e-07,0.166398
0,total_rooms,0.157801,0.000000e+00,0.162451
10,bath_log1p,0.264670,0.000000e+00,0.133422
9,bed_log1p,0.071496,6.798531e-225,0.108659


The table above reports the Pearson correlation, its p-value, and the mutual
information for each engineered feature. Features with **low p-values**
(e.g. < 0.05) and **high mutual information** are more likely to exert a real
influence on `price` rather than reflecting random noise.

---
# 6. Feature Selection and Raw-Parent Dropping

In this section, we perform feature selection and decide, for each engineered
feature, whether it should **replace** its raw parents.

---

## 6.1 F-Test for Regression

For feature matrix \( X \) and target \( y \), `f_regression` computes an
F-statistic for each feature, testing:

- \( H_0 \): feature is not linearly related to the target
- \( H_1 \): feature is linearly related to the target

We combine:

- F-statistics and their p-values
- Mutual information scores

to rank features and make decisions on which ones to retain.

In [ ]:
def build_feature_matrix(
    df: pd.DataFrame,
    target_col: str,
    engineered_cols: List[str],
) -> Tuple[pd.DataFrame, pd.Series]:
    """Construct the numeric feature matrix for modelling.

    This function consolidates raw numeric features and engineered features
    into a unified feature matrix suitable for statistical selection or
    downstream modelling. It excludes the target column from ``X`` and
    performs light cleaning on numeric fields (handling infinities and
    imputing missing values).

    Args:
        df: Input DataFrame containing raw and engineered features.
        target_col: Name of the supervised learning target variable.
        engineered_cols: List of engineered feature names that *must* be
            retained in the candidate matrix even if they are not naturally
            numeric columns within the dataset.

    Returns:
        A tuple containing:
            * X_candidates: A DataFrame of numeric candidate features.
            * y: The target Series aligned to ``X_candidates``.
    """
    # Select all numeric columns except the target
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in num_cols:
        num_cols.remove(target_col)

    # Ensure engineered numeric features are always included
    for col in engineered_cols:
        if col not in num_cols:
            num_cols.append(col)

    # Extract numeric feature matrix
    X_candidates = df[num_cols].copy()
    y = df[target_col].copy()

    # Replace infinite values and impute remaining NaNs
    X_candidates = X_candidates.replace([np.inf, -np.inf], np.nan)
    X_candidates = X_candidates.fillna(
        X_candidates.median(numeric_only=True)
    )

    return X_candidates, y


# ---- Build Feature Matrix ----
X_candidates, y = build_feature_matrix(
    df_fe,
    TARGET_COL,
    engineered_cols,
)

print(f"[INFO] Candidate feature matrix shape: {X_candidates.shape}")

[INFO] Candidate feature matrix shape: (2224841, 21)


We have assembled a candidate numeric feature matrix that includes both raw and
engineered features, excluding the target. Missing values, if any, are filled
using the median to keep the statistical procedures stable.

---
## 6.2 F-Test and Mutual Information Ranking

Next, we compute:

- `f_regression` for each feature (F-statistic and p-value)
- `mutual_info_regression` for non-linear dependency

We use a sampled subset to keep the runtime reasonable.

In [ ]:
def run_univariate_feature_screening(
    X: pd.DataFrame,
    y: pd.Series,
    max_sample: int = MAX_SAMPLE_FOR_STATS,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Run univariate feature screening using F-test and mutual information.

    This function:
      * Optionally subsamples the data for speed.
      * Standardises features for F-test stability.
      * Computes F-statistics and p-values via ``f_regression``.
      * Computes mutual information scores via ``mutual_info_regression``.
      * Returns a ranking of features sorted by mutual information.

    Args:
        X: Candidate numeric feature matrix.
        y: Target series aligned with ``X``.
        max_sample: Maximum number of rows to use for screening. If the
            dataset exceeds this size, a stratified (by index only) sample
            is drawn using ``train_size=max_sample``.
        random_state: Random seed for reproducible sampling.

    Returns:
        A DataFrame where each row represents a feature and includes:
            * ``feature``: Feature name.
            * ``F_stat``: F-statistic from the linear regression F-test.
            * ``F_pvalue``: Corresponding p-value for the F-test.
            * ``mutual_info``: Mutual information between the feature and
              the target.
        The DataFrame is sorted in descending order of ``mutual_info``.
    """
    # Optional downsampling for speed
    if len(X) > max_sample:
        X_fs, _, y_fs, _ = train_test_split(
            X,
            y,
            train_size=max_sample,
            random_state=random_state,
        )
    else:
        X_fs, y_fs = X.copy(), y.copy()

    # Standardise features for F-test stability
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_fs)

    # F-test
    F_vals, p_vals = f_regression(X_scaled, y_fs)

    # Mutual information (on original scale)
    mi_vals_all = mutual_info_regression(
        X_fs.values,
        y_fs.values,
        random_state=random_state,
    )

    fs_summary = pd.DataFrame(
        {
            "feature": X_fs.columns,
            "F_stat": F_vals,
            "F_pvalue": p_vals,
            "mutual_info": mi_vals_all,
        }
    ).sort_values("mutual_info", ascending=False)

    return fs_summary


# ---- Run Univariate Feature Screening ----
fs_summary = run_univariate_feature_screening(
    X_candidates,
    y,
    max_sample=MAX_SAMPLE_FOR_STATS,
    random_state=RANDOM_STATE,
)

fs_summary.head(20)

,feature,F_stat,F_pvalue,mutual_info
3,house_size,9793.717770,0.000000e+00,0.247064
12,house_size_log1p,11055.831327,0.000000e+00,0.246905
7,house_size_per_bed,15571.277355,0.000000e+00,0.209756
9,house_size_per_room,6445.621071,0.000000e+00,0.196696
17,state_listing_count,5267.696627,0.000000e+00,0.186629
16,city_listing_count,1045.582659,8.690022e-229,0.181325
11,building_coverage_ratio,2011.053058,0.000000e+00,0.168906
4,total_rooms,5030.886444,0.000000e+00,0.166498
8,house_size_per_bath,14.922764,1.120386e-04,0.163459
14,bath_log1p,14721.068853,0.000000e+00,0.136657


The feature selection summary above shows the relative strength of each
candidate feature according to the F-test and mutual information. We now use
this information, alongside the `parent_map`, to decide whether engineered
features should replace their raw parents.

---
## 6.3 Resolving Engineered Features vs Raw Parents

For each engineered feature:

1. Retrieve its mutual information score.
2. Retrieve the mutual information scores for its parent columns.
3. If the engineered feature has **higher mutual information than all parents**,
   we prefer the engineered version and drop its parents from the final feature
   set.
4. Otherwise, we retain the parents and drop the engineered feature.

This ensures that we never keep both the raw and engineered versions of the
same underlying information, reducing redundancy and potential multicollinearity.

In [ ]:
def decide_feature_retention(
    fs_table: pd.DataFrame,
    parent_map: Dict[str, List[str]],
) -> Tuple[List[str], List[str]]:
    """Resolve which features to keep, comparing engineered vs parent features.

    This function uses mutual information scores from ``fs_table`` and the
    lineage information in ``parent_map`` to decide whether to keep an
    engineered feature, its parents, or both. The core heuristic is:

        • If an engineered feature's mutual information is greater than or
          equal to *all* of its parents' mutual information scores, the
          engineered feature is preferred and the parents are dropped.
        • Otherwise, the parents are preferred and the engineered feature is
          dropped.

    Features not present in ``parent_map`` are left untouched and follow the
    feature selection table as-is.

    Args:
        fs_table: Feature selection summary with at least the following
            columns:
                - ``feature``
                - ``mutual_info``
              (F-statistics and p-values may also be present but are not used
              in this decision logic.)
        parent_map: Mapping from engineered feature name to a list of its
            parent raw column names.

    Returns:
        A tuple of:
            keep_features: Sorted list of feature names to retain.
            drop_features: Sorted list of feature names to drop.
    """
    mi_lookup = fs_table.set_index("feature")["mutual_info"].to_dict()
    all_features = set(fs_table["feature"].tolist())

    drop_features: List[str] = []
    prefer_engineered: List[str] = []

    for eng_feat, parents in parent_map.items():
        # Only consider engineered features that survived univariate screening
        if eng_feat not in all_features:
            continue

        parent_in_fs = [parent for parent in parents if parent in all_features]
        if not parent_in_fs:
            # No parent in feature set → keep engineered by default
            continue

        eng_mi = mi_lookup.get(eng_feat, 0.0)
        parent_mis = [mi_lookup.get(parent, 0.0) for parent in parent_in_fs]

        if all(eng_mi >= mi for mi in parent_mis):
            # Engineered feature dominates → drop parents
            drop_features.extend(parent_in_fs)
            prefer_engineered.append(eng_feat)
        else:
            # Parents are as good or better → drop engineered
            drop_features.append(eng_feat)

    keep_features = sorted(list(all_features.difference(drop_features)))
    drop_features = sorted(list(set(drop_features)))

    return keep_features, drop_features


# ---- Apply Feature Retention Decision ----
keep_feats, drop_feats = decide_feature_retention(fs_summary, parent_map)

print("[INFO] Features to DROP (raw or engineered):")
print(drop_feats)

print("\n[INFO] Features to KEEP (numeric, first 40 shown):")
print(keep_feats[:40], "...")

print(
    f"\n[INFO] Total kept: {len(keep_feats)}  |  "
    f"Total dropped: {len(drop_feats)}"
)

[INFO] Features to DROP (raw or engineered):
['acre_lot', 'bath', 'bath_per_bed', 'bed', 'bed_log1p', 'bed_per_bath', 'building_coverage_ratio', 'house_size_log1p', 'house_size_per_bath', 'house_size_per_bed', 'house_size_per_room']

[INFO] Features to KEEP (numeric, first 40 shown):
['acre_lot_log1p', 'bath_log1p', 'city_listing_count', 'house_size', 'lot_sqft', 'state_listing_count', 'status_is_for_sale', 'status_is_ready_to_build', 'status_is_sold', 'total_rooms'] ...

[INFO] Total kept: 10  |  Total dropped: 11


The lists above summarise which features will be **retained** and which will be
**dropped**. In particular, wherever an engineered feature clearly outperformed
its parents (higher mutual information), the parents were dropped; otherwise the
engineered feature itself was removed.

This directly addresses the earlier pitfall of accidentally keeping both raw
and engineered versions of the same information.

---
## 7. Build Final Feature Matrix for Modelling

Finally, we construct the modelling-ready feature matrix `X_final` based on the
selected features. This matrix can be persisted to disk or passed directly into
Stage 05 (modelling).


In [ ]:
def build_final_feature_matrix(
    X_candidates: pd.DataFrame,
    y: pd.Series,
    keep_features: List[str],
) -> Tuple[pd.DataFrame, pd.Series]:
    """Construct the final feature matrix for modelling.

    This function:
      * Filters the candidate matrix down to the selected features.
      * Handles one-hot-encoded status columns by dropping one category to
        avoid the dummy-variable trap.
      * Drops redundant unit-conversion features (e.g., ``lot_sqft`` when
        ``acre_lot`` and/or ``acre_lot_log1p`` are present).

    Args:
        X_candidates: Candidate feature matrix prior to final pruning.
        y: Target series aligned with ``X_candidates``.
        keep_features: List of feature names that passed the retention
            decision step.

    Returns:
        A tuple of:
            * X_final: Final feature matrix for modelling.
            * y_final: Target series (unchanged, but aligned to ``X_final``).
    """
    # Step 1: Base X and y
    X_final = X_candidates[keep_features].copy()
    y_final = y.copy()

    # Step 2: Handle one-hot-encoded 'status' columns safely
    status_cols = [
        col for col in X_final.columns
        if col.startswith("status_is_")
    ]

    if len(status_cols) > 1:
        status_cols_sorted = sorted(status_cols)
        col_to_drop = status_cols_sorted[0]  # deterministic drop
        print(
            "[INFO] Dropping one-hot category to avoid dummy trap: "
            f"{col_to_drop}"
        )
        X_final = X_final.drop(columns=[col_to_drop])

    # Step 3: Drop redundant unit-conversion features
    redundant_groups = {
        # ``lot_sqft`` is strictly a scaled version of ``acre_lot``.
        "acre_lot": ["lot_sqft"],
        # Add more mappings here if needed.
    }

    for base, redundant_list in redundant_groups.items():
        log_col = f"{base}_log1p"

        for redundant_col in redundant_list:
            if redundant_col in X_final.columns:
                # Only drop if base and/or its log-transformed version exist
                if base in X_final.columns or log_col in X_final.columns:
                    print(
                        "[INFO] Dropping redundant unit-converted feature: "
                        f"{redundant_col}"
                    )
                    X_final = X_final.drop(columns=[redundant_col])

    print(f"[INFO] Final X shape: {X_final.shape}")
    print(f"[INFO] y length: {len(y_final)}")

    return X_final, y_final


# ---- Build Final Feature Matrix ----
X_final, y_final = build_final_feature_matrix(
    X_candidates=X_candidates,
    y=y,
    keep_features=keep_feats,
)

X_final.head()

[INFO] Dropping one-hot category to avoid dummy trap: status_is_for_sale
[INFO] Dropping redundant unit-converted feature: lot_sqft
[INFO] Final X shape: (2224841, 8)
[INFO] y length: 2224841


,acre_lot_log1p,bath_log1p,city_listing_count,house_size,state_listing_count,status_is_ready_to_build,status_is_sold,total_rooms
0,0.113329,1.098612,2,920.0,3126,0,0,5
1,0.076961,1.098612,2,1527.0,3126,0,0,6
2,0.139762,0.693147,17,748.0,3126,0,0,3
3,0.095310,1.098612,81,1800.0,3126,0,0,6
4,0.048790,1.098612,64,2514.0,3126,0,0,8


The final feature matrix `X_final` now contains only the selected features,
with raw parents removed where appropriate and engineered features retained
when they demonstrated stronger association with `price`. This dataset is now
ready to be consumed by the modelling pipeline in the next stage.

---
# 8. Exporting Cleaned Dataset

Finally, we persist the dataset for downstream use. We save in both:

- **Parquet**: efficient binary format for Python-based pipelines.
- **CSV**: convenient for quick inspection or use in non-Python tools.

The paths follow the project's `/data_processed/` convention.

In [ ]:
def export_final_datasets(
    X_final: pd.DataFrame,
    y_final: pd.Series,
    output_dir: str = "data_processed",
) -> Dict[str, str]:
    """Export the final modelling datasets in both Parquet and CSV formats.

    This function:
      * Ensures the export directory exists.
      * Applies a timestamp to filenames for reproducibility and versioning.
      * Saves:
            - ``X_final`` (features only)
            - ``y_final`` (target only)
            - ``combined`` (features + target)
        in both Parquet and CSV formats.

    Args:
        X_final: Final feature matrix post-selection and redundancy removal.
        y_final: Target series aligned with ``X_final``.
        output_dir: Directory where exported files will be written.

    Returns:
        A dictionary containing the filepaths of all exported files.
    """
    os.makedirs(output_dir, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    # Construct all filepaths
    X_parquet_path = os.path.join(output_dir, f"X_final_{timestamp}.parquet")
    y_parquet_path = os.path.join(output_dir, f"y_final_{timestamp}.parquet")
    combined_parquet_path = os.path.join(
        output_dir, f"final_dataset_{timestamp}.parquet"
    )

    X_csv_path = os.path.join(output_dir, f"X_final_{timestamp}.csv")
    y_csv_path = os.path.join(output_dir, f"y_final_{timestamp}.csv")
    combined_csv_path = os.path.join(
        output_dir, f"final_dataset_{timestamp}.csv"
    )

    # Build combined DataFrame
    df_final = X_final.copy()
    df_final["price"] = y_final

    # --- Parquet Exports ---
    X_final.to_parquet(X_parquet_path, index=False)
    y_final.to_frame().to_parquet(y_parquet_path, index=False)
    df_final.to_parquet(combined_parquet_path, index=False)

    # --- CSV Exports ---
    X_final.to_csv(X_csv_path, index=False)
    y_final.to_frame().to_csv(y_csv_path, index=False)
    df_final.to_csv(combined_csv_path, index=False)

    # Build return dictionary
    return {
        "X_parquet": X_parquet_path,
        "y_parquet": y_parquet_path,
        "combined_parquet": combined_parquet_path,
        "X_csv": X_csv_path,
        "y_csv": y_csv_path,
        "combined_csv": combined_csv_path,
    }


# ---- Execute Export ----
export_paths = export_final_datasets(X_final, y_final)

print("[INFO] Export Complete!")
for label, path in export_paths.items():
    print(f"  - {label}: {path}")

[INFO] Export Complete!
  - X_parquet: data_processed/X_final_20251115_1831.parquet
  - y_parquet: data_processed/y_final_20251115_1831.parquet
  - combined_parquet: data_processed/final_dataset_20251115_1831.parquet
  - X_csv: data_processed/X_final_20251115_1831.csv
  - y_csv: data_processed/y_final_20251115_1831.csv
  - combined_csv: data_processed/final_dataset_20251115_1831.csv


The cleaned dataset is now available under `/data_processed/` and can be
safely used by the next notebook (`04_visualisation`) without
repeating any heavy engineering logic. The feature engineering pipeline can later be refactored into a Python module and wired into a full MLOps workflow if required.